In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/)
import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import torch  # import torch first to avoid circular import
from kaggle_secrets import UserSecretsClient
import wandb

# Get API key from Kaggle Secrets
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB_API_KEY")

# Login to wandb
wandb.login(key=wandb_key)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [3]:
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
import lightgbm as lgb
from sentence_transformers import CrossEncoder, InputExample
import wandb

# Set seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# MILESTONE 4

Multiple-Choice Data Formatting
In this section, you will convert the Kaggle MCQ format into the structure required by multiple-choice models. Each question has one prompt and five options and each option must be paired with the prompt separately.

# Q1. Label Encoding
Convert the answer column in train.csv into numeric labels using the following mapping:
A = 0
B = 1
C = 2
D = 3
E = 4

# What is the encoded numeric label for the row at index 150?


In [3]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

# Mapping of choices to numeric labels
label_mapping = {
    'A': 0,
    'B': 1,
    'C': 2,
    'D': 3,
    'E': 4
}

# Convert the answer column into numeric labels
df['label'] = df['answer'].map(label_mapping)

# Retrieve the encoded numeric label for the row at index 150
encoded_label_150 = df.loc[150, 'label']
print(f"Encoded label at index 150: {encoded_label_150}")

Encoded label at index 150: 2


# Q2. Prompt-Option Formatting
For row index 0, create the Option B input using exactly this format:
str(prompt) + " [SEP] " + str(option_B)

# What is the exact character length of this formatted input string?


In [4]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

# Get the first row (index 0)
row = df.iloc[0]

# Format the input string for Option B
formatted_input = str(row['prompt']) + " [SEP] " + str(row['B'])

# Calculate and print the exact character length
print("Formatted string:")
print(formatted_input)
print(f"Exact character length: {len(formatted_input)}")

Formatted string:
Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
Exact character length: 407


# Tokenization for Multiple-Choice Models
Multiple-choice models expect inputs in the shape:
batch_size x num_choices x sequence_length

Since each question has five options, every row becomes five tokenized sequences.

# Q3. Single-Row MCQ Tokenization
Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:
padding = "max_length"
truncation = True
max_length = 128
return_tensors = "pt"

After reshaping for a multiple-choice model, the final input_ids tensor has shape:
[1, 5, 128]

# What is the value of the second dimension?

In [5]:
import pandas as pd
import torch
from transformers import AutoTokenizer

# 1. Load data
df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
row = df.iloc[0]

# 2. Extract prompt and options
prompt = row['prompt']
options = [row['A'], row['B'], row['C'], row['D'], row['E']]

# 3. Format inputs: prompt + [SEP] + option
formatted_inputs = [str(prompt) + " [SEP] " + str(opt) for opt in options]

# 4. Initialize tokenizer and tokenize
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tokenized = tokenizer(
    formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# 5. Reshape to include batch dimension
# input_ids initially has shape [5, 128]
# unsqueeze(0) reshapes it to [1, 5, 128]
input_ids = tokenized['input_ids'].unsqueeze(0)

print("Shape of input_ids tensor:", list(input_ids.shape))
print("Value of the second dimension (num_choices):", input_ids.shape[1])

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Shape of input_ids tensor: [1, 5, 128]
Value of the second dimension (num_choices): 5


# Q4. Batch MCQ Tokenization
Tokenize the first 16 rows of train.csv as multiple-choice examples.
Each row has 5 choices.
Each choice is tokenized to length 128.

The final input_ids tensor has shape:
[16, 5, 128]

# How many total token positions are in this tensor?

In [6]:
import pandas as pd
import torch
from transformers import AutoTokenizer

# 1. Load data and select the first 16 rows
df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
subset = df.iloc[:16]

# 2. Extract and format the prompt-option pairs for all 16 rows
formatted_list = []
for idx, row in subset.iterrows():
    prompt = row['prompt']
    options = [row['A'], row['B'], row['C'], row['D'], row['E']]
    formatted_list.extend([str(prompt) + " [SEP] " + str(opt) for opt in options])

# 3. Tokenize the entire batch
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tokenized = tokenizer(
    formatted_list,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# 4. Reshape from [16 * 5, 128] to [16, 5, 128]
input_ids = tokenized['input_ids'].view(16, 5, 128)

# 5. Print the shape and total token positions (elements)
print("Tensor Shape:", list(input_ids.shape))
print("Total Token Positions (numel):", input_ids.numel())

Tensor Shape: [16, 5, 128]
Total Token Positions (numel): 10240


# Multiple-Choice Model Outputs
# AutoModelForMultipleChoice produces one logit score for each answer option. For this competition, the model outputs five logits corresponding to A, B, C, D, and E.

# Q5. Multiple-Choice Logits
Load bert-base-uncased using AutoModelForMultipleChoice.
Tokenize row index 0 as 5 choices and pass it through the model.

The output logits tensor has shape:
[1, 5]

# How many logits are produced for one question?

In [7]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice

# 1. Load the first row of train.csv
df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
row = df.iloc[0]

# 2. Prepare inputs (prompt paired with each of the 5 options)
prompt = row['prompt']
options = [row['A'], row['B'], row['C'], row['D'], row['E']]
formatted_inputs = [str(prompt) + " [SEP] " + str(opt) for opt in options]

# 3. Tokenize inputs
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
inputs = tokenizer(
    formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# 4. Add batch dimension: [5, 128] -> [1, 5, 128]
inputs = {k: v.unsqueeze(0) for k, v in inputs.items()}

# 5. Load model and pass the inputs to get outputs
model = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')
with torch.no_grad():
    outputs = model(**inputs)

# 6. Retrieve logits and shape information
logits = outputs.logits
print("Logits Shape:", list(logits.shape))
print("Number of logits for one question:", logits.shape[1])

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Logits Shape: [1, 5]
Number of logits for one question: 5


# Q6. Supervised Loss Tensor
For row index 0, pass the tokenized 5-choice input into AutoModelForMultipleChoice along with the correct encoded label.

The model returns a scalar loss tensor.

# How many dimensions does this loss tensor have?

In [8]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice

# 1. Load the first row of train.csv
df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
row = df.iloc[0]

# 2. Get the correct label (option 'B' at index 0 maps to label 1)
label_mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
label = torch.tensor([label_mapping[row['answer']]]) # shape: [1]

# 3. Format inputs (prompt paired with each of the 5 options)
prompt = row['prompt']
options = [row['A'], row['B'], row['C'], row['D'], row['E']]
formatted_inputs = [str(prompt) + " [SEP] " + str(opt) for opt in options]

# 4. Tokenize inputs and add batch dimension
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
inputs = tokenizer(
    formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)
inputs = {k: v.unsqueeze(0) for k, v in inputs.items()}

# 5. Add label to the inputs dictionary
inputs['labels'] = label

# 6. Load model and pass the inputs to compute loss
model = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')
with torch.no_grad():
    outputs = model(**inputs)

# 7. Print loss tensor details
loss = outputs.loss
print("Loss Tensor:", loss)
print("Loss Tensor Shape:", list(loss.shape))
print("Number of Dimensions (ndim):", loss.ndim)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loss Tensor: tensor(1.5935)
Loss Tensor Shape: []
Number of Dimensions (ndim): 0


# LoRA for Efficient Fine-Tuning
# LoRA freezes most of the original model and trains only a small number of adapter parameters. This makes fine-tuning faster and more memory-efficient.

# Q7. LoRA Trainable Parameters
# Apply LoRA to the bert-base-uncased multiple-choice model using:
r = 8
lora_alpha = 16
target_modules = ["query", "value"]
lora_dropout = 0.1
bias = "none"
task_type = TaskType.SEQ_CLS

Count trainable parameters using:
sum(p.numel() for p in model.parameters() if p.requires_grad)

# How many parameters are trainable?

In [9]:
from transformers import AutoModelForMultipleChoice
from peft import LoraConfig, get_peft_model, TaskType

# 1. Load the model architecture
model = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')

# 2. Define the LoRA Configuration
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none"
)

# 3. Apply the LoRA adapter wrapper
peft_model = get_peft_model(model, peft_config)

# 4. Count the number of trainable parameters
trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)

print(f"Trainable parameters: {trainable_params}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable parameters: 295681


# Preparing Data for Hugging Face Trainer
# Before training, the dataset must be converted into a format that the Hugging Face Trainer can understand: tokenized input_ids, attention_mask, and numeric labels.

# Q8. Hugging Face Dataset Preparation
# Create a Hugging Face Dataset from the first 100 rows of train.csv.

For each row, create:
input_ids with shape [5, 128]
attention_mask with shape [5, 128]
labels as the encoded answer label

For the first dataset item, input_ids has shape:
[5, 128]

# How many tokenized choices are stored in input_ids?

In [11]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer

# 1. Load the dataset (first 100 rows)
df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv').head(100)

# 2. Initialize tokenizer and mapping
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
label_mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

# 3. Define the preprocessing function
def preprocess_function(examples):
    num_rows = len(examples['prompt'])
    input_ids_list = []
    attention_mask_list = []
    labels_list = []
    
    options_keys = ['A', 'B', 'C', 'D', 'E']
    
    for i in range(num_rows):
        prompt = examples['prompt'][i]
        options = [examples[key][i] for key in options_keys]
        formatted_inputs = [str(prompt) + " [SEP] " + str(opt) for opt in options]
        
        tokenized = tokenizer(
            formatted_inputs,
            padding="max_length",
            truncation=True,
            max_length=128,
        )
        
        input_ids_list.append(tokenized['input_ids'])
        attention_mask_list.append(tokenized['attention_mask'])
        labels_list.append(label_mapping[examples['answer'][i]])
        
    return {
        'input_ids': input_ids_list,
        'attention_mask': attention_mask_list,
        'label': labels_list
    }

# 4. Build and tokenize the Hugging Face Dataset
dataset = Dataset.from_pandas(df)
tokenized_dataset = dataset.map(preprocess_function, batched=True)

# 5. Inspect the first item
first_item = tokenized_dataset[0]
first_input_ids = first_item['input_ids']

print("Shape of input_ids in first item:", [len(first_input_ids), len(first_input_ids[0])])
print("Number of tokenized choices stored in input_ids:", len(first_input_ids))

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Shape of input_ids in first item: [5, 128]
Number of tokenized choices stored in input_ids: 5


# Tiny Fine-Tuning and Inference
# In this section, you will run a very small LoRA fine-tuning job using Hugging Face Trainer. Then you will use the fine-tuned model to produce probabilities for the answer options.

# Q9. Tiny LoRA Fine-Tuning
# Fine-tune a LoRA multiple-choice model on the first 32 rows using Hugging Face Trainer.

Use the following settings:
max_length = 64
per_device_train_batch_size = 4
gradient_accumulation_steps = 1
max_steps = 4

# What is the final global_step reported by the Trainer?

In [12]:
import dataclasses
from typing import Dict, List, Any
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType

# 1. Load the first 32 rows of train.csv
df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv').head(32)

# 2. Initialize tokenizer and mapping
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
label_mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

# 3. Define the preprocessing function
def preprocess_function(examples):
    num_rows = len(examples['prompt'])
    input_ids_list = []
    attention_mask_list = []
    labels_list = []
    
    options_keys = ['A', 'B', 'C', 'D', 'E']
    
    for i in range(num_rows):
        prompt = examples['prompt'][i]
        options = [examples[key][i] for key in options_keys]
        formatted_inputs = [str(prompt) + " [SEP] " + str(opt) for opt in options]
        
        tokenized = tokenizer(
            formatted_inputs,
            padding="max_length",
            truncation=True,
            max_length=64, # max_length set to 64
        )
        
        input_ids_list.append(tokenized['input_ids'])
        attention_mask_list.append(tokenized['attention_mask'])
        labels_list.append(label_mapping[examples['answer'][i]])
        
    return {
        'input_ids': input_ids_list,
        'attention_mask': attention_mask_list,
        'label': labels_list
    }

# Convert to Hugging Face Dataset
dataset = Dataset.from_pandas(df)
tokenized_dataset = dataset.map(preprocess_function, batched=True)

# 4. Load the base model and apply LoRA
model = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none"
)
model = get_peft_model(model, peft_config)

# 5. Create a custom data collator for multiple-choice data
@dataclasses.dataclass
class DataCollatorForMultipleChoice:
    tokenizer: Any
    
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        label_name = "label" if "label" in features[0].keys() else "labels"
        labels = [feature.pop(label_name) for feature in features]
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        
        flat_features = []
        for feature in features:
            for i in range(num_choices):
                flat_features.append({
                    "input_ids": feature["input_ids"][i],
                    "attention_mask": feature["attention_mask"][i]
                })
        
        batch = self.tokenizer.pad(
            flat_features,
            padding=True,
            return_tensors="pt",
        )
        
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch

# 6. Set Training Arguments
training_args = TrainingArguments(
    output_dir="./lora_tuning_output",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4, # specified setting
    logging_steps=1,
    report_to="none"
)

# 7. Initialize Trainer and run training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForMultipleChoice(tokenizer=tokenizer),
)

print("Starting training...")
train_result = trainer.train()
print(f"Final global_step reported by the Trainer: {train_result.global_step}")

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
1,3.234903
2,3.258008
3,3.267799
4,3.187373


Final global_step reported by the Trainer: 4


# Q10. Probability Assigned to Option E After Fine-Tuning
# Using the fine-tuned LoRA model from Q9, run inference on row index 0 and apply softmax to the logits.

# What is the probability assigned to Option E?

# Round your answer to 4 decimal places.

In [13]:
import dataclasses
from typing import Dict, List, Any
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer, set_seed
from peft import LoraConfig, get_peft_model, TaskType

# Set seed for reproducibility
set_seed(42)

# 1. Load the first 32 rows of train.csv
df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv').head(32)

# 2. Initialize tokenizer and mapping
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
label_mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

# 3. Define the preprocessing function
def preprocess_function(examples):
    num_rows = len(examples['prompt'])
    input_ids_list = []
    attention_mask_list = []
    labels_list = []
    
    options_keys = ['A', 'B', 'C', 'D', 'E']
    
    for i in range(num_rows):
        prompt = examples['prompt'][i]
        options = [examples[key][i] for key in options_keys]
        formatted_inputs = [str(prompt) + " [SEP] " + str(opt) for opt in options]
        
        tokenized = tokenizer(
            formatted_inputs,
            padding="max_length",
            truncation=True,
            max_length=64, # specified setting
        )
        
        input_ids_list.append(tokenized['input_ids'])
        attention_mask_list.append(tokenized['attention_mask'])
        labels_list.append(label_mapping[examples['answer'][i]])
        
    return {
        'input_ids': input_ids_list,
        'attention_mask': attention_mask_list,
        'label': labels_list
    }

# Convert to Hugging Face Dataset
dataset = Dataset.from_pandas(df)
tokenized_dataset = dataset.map(preprocess_function, batched=True)

# 4. Load the base model and apply LoRA
model = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none"
)
model = get_peft_model(model, peft_config)

# 5. Create custom multiple-choice data collator
@dataclasses.dataclass
class DataCollatorForMultipleChoice:
    tokenizer: Any
    
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        label_name = "label" if "label" in features[0].keys() else "labels"
        labels = [feature.pop(label_name) for feature in features]
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        
        flat_features = []
        for feature in features:
            for i in range(num_choices):
                flat_features.append({
                    "input_ids": feature["input_ids"][i],
                    "attention_mask": feature["attention_mask"][i]
                })
        
        batch = self.tokenizer.pad(
            flat_features,
            padding=True,
            return_tensors="pt",
        )
        
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch

# 6. Training Arguments
training_args = TrainingArguments(
    output_dir="./lora_tuning_output",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    report_to="none",
    seed=42
)

# 7. Initialize Trainer and Train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForMultipleChoice(tokenizer=tokenizer),
)
trainer.train()

# 8. Inference on row index 0
row_0 = tokenized_dataset[0]

input_ids = torch.tensor(row_0['input_ids']).unsqueeze(0)
attention_mask = torch.tensor(row_0['attention_mask']).unsqueeze(0)

# Move inputs to the model device
device = next(model.parameters()).device
input_ids = input_ids.to(device)
attention_mask = attention_mask.to(device)

model.eval()
with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits
    probs = F.softmax(logits, dim=-1)

prob_E = probs[0, 4].item()
print(f"Probability of Option E: {prob_E:.4f}")

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch

Step,Training Loss
1,3.135577
2,3.244414
3,3.215415
4,3.217349


Probability of Option E: 0.1967
